In [1]:
import pandas as pd
import numpy as np

In [2]:
def read_data(path , sheet_name=None):
    xls = pd.ExcelFile(path)
    # Read the sheet into a DataFrame
    sheet_data = pd.read_excel(xls, sheet_name=sheet_name, header=None)
    # Extract relevant columns (assumed to have certain keywords, could vary in real sheets)
    # Here we use placeholder for the actual extraction process
    relevant_data = sheet_data.copy()  # This is where extraction logic will be applied
    return xls, relevant_data

In [3]:
# Đọc dữ liệu từ file Excel
file_path = 'D:/Work/VHL/VHL_Biology/data/Metadata/GGA-metal/File Excel/Excel-GGA-metal HH.xlsx'  # Thay bằng đường dẫn thực tế đến file Excel của bạn
xls, sheets = read_data(file_path)


In [4]:
lst_sheets = xls.sheet_names

In [5]:
def find_sample(sheet_name, sample):
    df = sheets[sheet_name]
    result = df.isin([sample])
    if result.any().any():
        # Dùng 'stack' để tìm tất cả vị trí của giá trị trong DataFrame
        positions = list(zip(*np.where(result)))
        
        # Hiển thị tất cả các vị trí tìm thấy
        for row, col in positions:
            print(f"Value found at row {row}, column {df.columns[col]}")
    else:
        print("Value not found in the table.")
    return int(row), int(col)

In [6]:
# assert find_sample(lst_sheets[0],'27032024-BOD-15-10-Q=49.66mL/phút-1') == (0,4)

In [7]:
# Re-define the function to process sheets based on the observed structure in the image
def process_sheet(sheet_name):
    df_sheet = pd.DataFrame(columns=['Tag', 'Doin (mV)', 'No.peak', 'DOmin (mV)', 'DDO (mV)','Sheet Name', 'Sample Name'])
    _,relevant_data = read_data(file_path,sheet_name)
    relevant_data = relevant_data.fillna(" ").iloc[:,1:]
    lst_samples=[]
    # Loop through the rows of the sheet
    for idx, row in relevant_data.iterrows():
        # Check for the presence of a sample name in the row (based on pattern in columns like 'U43-H1-VS2...')
        for col in row:
            if isinstance(col, str) and 'Q' in col:  # Pattern to match sample names
                lst_samples.append(col)

    for sample in lst_samples:
        _, cols = find_sample(sheet_name,sample)
        # Get the table in sheet base on sample name's column 
        min_col=cols - 1
        max_col=cols + 3
        dct = {}
        len_data = 0
        for col in range(max_col,min_col-1, -1):
            lst = []
            if col == min_col:
                start_row = relevant_data[col].loc[relevant_data[col].str.strip() != ""].first_valid_index()
                num_rows = len_data
                subset = relevant_data[col].iloc[start_row:start_row+num_rows].replace(" ", np.nan).ffill()
                lst =  subset.tolist()
            for value in relevant_data[col]:
                if not (isinstance(value, float) or isinstance(value, int)): 
                    pass
                else:
                    lst.append(value)
            dct[col]=lst
            len_data = len(lst)

        sorted_dct = dict(sorted(dct.items()))
        target_names = ['Tag', 'Doin (mV)', 'No.peak', 'DOmin (mV)', 'DDO (mV)']
        keys = list(sorted_dct.keys())
        i=0
        for key in keys:
            sorted_dct[target_names[i]]= sorted_dct.pop(key)
            i+=1

        df_sample = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in sorted_dct.items()]))
        df_sample['Sheet Name']= sheet_name
        df_sample['Sample Name']= sample

        # Loại bỏ các cột chứa toàn bộ NaN
        # df_sample = df_sample.dropna().infer_objects()
        # df_sheet = df_sheet.dropna()
        # Append the extracted information to the consolidated data
        df_sheet = pd.concat([df_sheet.astype(df_sample.dtypes), df_sample.astype(df_sheet.dtypes)], ignore_index=True)
        print(f"Sheet {sheet_name}\nSample {sample}")
    
    return df_sheet


In [8]:
# Initialize an empty DataFrame to hold the consolidated data
consolidated_data = pd.DataFrame(columns=['Tag', 'Doin (mV)', 'No.peak', 'DOmin (mV)', 'DDO (mV)','Sheet Name', 'Sample Name'])

# Process each sheet again with the updated logic
for sheet in lst_sheets:
    df_sheet = process_sheet(sheet)
    # consolidated_data = consolidated_data.dropna()
    consolidated_data = pd.concat([consolidated_data.astype(df_sheet.dtypes), df_sheet.astype(consolidated_data.dtypes)], ignore_index=True)
    
# Show the updated consolidated data
consolidated_data


Value found at row 0, column 3
Sheet U183-H5-VS2-03.09.2024
Sample U183-H5-VS2-GGA2.5-HH 5mg-L-06092024-Q=50,04mL-phút-1
Value found at row 0, column 3
Sheet U184-H5-VS2-03.09.2024
Sample U184-H5-VS2-GGA5-HH 10mg-L-09092024-Q=49.68mL-phút-1
Value found at row 0, column 2
Sheet U185-H5-VS2-05.09.2024
Sample U185-H5-VS2-GGA5-HH 15mg-L-10092024-Q=50.4mL-phút-1
Value found at row 0, column 2
Sheet U187-H5-VS2-09.09.2024
Sample U187-H5-VS2-GGA7.5-HH 5mg-L-12092024-Q=50.12mL-phút-1
Value found at row 0, column 9
Sheet U187-H5-VS2-09.09.2024
Sample U187-H5-VS2-GGA7.5-HH 10mg-L-12092024-Q=50.18mL-phút-1
Value found at row 0, column 16
Sheet U187-H5-VS2-09.09.2024
Sample U187-H5-VS2-GGA2.5-HH 15mg-L-12092024-Q=50.32mL-phút-2
Value found at row 0, column 2
Sheet U188-H5-VS2-11.09.2024
Sample U188-H5-VS2-GGA7.5- Zn(II)5-Cr(VI)-10-Ni(II) 15mg-L-13092024-Q=50.24mL-phút-1
Value found at row 0, column 2
Sheet U189-H5-VS2-12.09.2024
Sample U189-H5-VS2-GGA7.5- HH 5mg-L-16092024-Q=50.3mL-phút-2
Value fo

,Tag,Doin (mV),No.peak,DOmin (mV),DDO (mV),Sheet Name,Sample Name
0,GGA2.5,299.96531,311,198.1,101.86531,U183-H5-VS2-03.09.2024,"U183-H5-VS2-GGA2.5-HH 5mg-L-06092024-Q=50,04mL..."
1,GGA2.5,295.03673,785,196.79999,98.23674,U183-H5-VS2-03.09.2024,"U183-H5-VS2-GGA2.5-HH 5mg-L-06092024-Q=50,04mL..."
2,GGA2.5,293.74082,1257,195.4,98.34082,U183-H5-VS2-03.09.2024,"U183-H5-VS2-GGA2.5-HH 5mg-L-06092024-Q=50,04mL..."
3,GGA2.5,292.506,1730,190.4,102.106,U183-H5-VS2-03.09.2024,"U183-H5-VS2-GGA2.5-HH 5mg-L-06092024-Q=50,04mL..."
4,GGA2.5,292.59592,2203,195.3,97.29592,U183-H5-VS2-03.09.2024,"U183-H5-VS2-GGA2.5-HH 5mg-L-06092024-Q=50,04mL..."
...,...,...,...,...,...,...,...
1453,GGA20-Zn15-Cr15-Ni10,294.52593,5038,273.40001,21.12592,U224-H6-VS2-16.10.2024,U224-H6-VS2-GGA20- Zn(II) 15-Cr(VI) 15-Ni(II) ...
1454,GGA20-Zn15-Cr15-Ni10,299.51296,5508,276.4,23.11296,U224-H6-VS2-16.10.2024,U224-H6-VS2-GGA20- Zn(II) 15-Cr(VI) 15-Ni(II) ...
1455,GGA20-Zn15-Cr15-Ni10,297.38519,5987,273.99999,23.3852,U224-H6-VS2-16.10.2024,U224-H6-VS2-GGA20- Zn(II) 15-Cr(VI) 15-Ni(II) ...
1456,GGA20-Zn15-Cr15-Ni10,293.68302,6457,272.50001,21.18301,U224-H6-VS2-16.10.2024,U224-H6-VS2-GGA20- Zn(II) 15-Cr(VI) 15-Ni(II) ...


In [9]:
save_path = 'D:/Work/VHL/VHL_Biology/data/'

In [10]:
from datetime import date
today = date.today()

In [11]:
consolidated_data.to_csv(save_path+f'metadata-gga-metal-HH-{today}.csv',encoding='utf-8-sig')